In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
# import name from utilities for bronze_schema etc ( need file extension also)
%run /Workspace/Users/thien1997nus@gmail.com/fmcg_databricks/src/1_setup/utilities.ipynb

In [0]:
# create widget so that can get data setup from it
# to specify environment when dev/prod, other datasources
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

In [0]:
## move files . use when wrong setup only
#dbutils.fs.mv("/Volumes/fmcg/default/fmcg/1_parent_company/full_load/dim_customers.csv", "/Volumes/fmcg/default/fmcg/1_parent_company/full_load/customers/dim_customers.csv")

In [0]:
# path data volumne
base_path = f'/Volumes/fmcg/default/{catalog}/2_child_company/full_load/{data_source}/*.csv'
print(base_path)





In [0]:
df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(base_path)
    .withColumn("read_timestamp", F.current_timestamp())
    .select( "*", "_metadata.file_name", "_metadata.file_size")
)
display(df.limit(10))

In [0]:
df.write\
    .format("delta")\
    .option( "delta.enableChangeDataFeed", "true") \
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

## Silver Processing

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")
display(df_bronze.limit(10))

In [0]:
df_bronze.printSchema()

In [0]:
df_duplicates = df_bronze.groupBy("customer_id").count().where("count > 1")
display(df_duplicates)

In [0]:
df_silver = df_bronze.dropDuplicates(["customer_id"])

In [0]:
# check lead space issue
display(
    df_silver.filter( F.col("customer_name") != F.trim( F.col("customer_name")) )
)

In [0]:
df_silver = df_silver.withColumn(
    "customer_name",
    F.trim( F.col("customer_name"))
)

In [0]:
df_silver.select('city').distinct().show()

In [0]:
# typo -> correct names
city_mapping = {
    'Bengaluruu' : 'Bengaluru',
    'Bengalore': 'Bengaluru',
    'Hyderabadd': 'Hyderabad',
    'Hyderabaddd': 'Hyderabad',
    'NewDelhi': 'New Delhi',
    'NewDheli': 'New Delhi',
    'NewDelhee': 'New Delhi'
}

allowed = ["Bengaluru", "Hyderabad", "New Delhi"]
df_silver = (
    df_silver
    .replace( city_mapping, subset= ["city"])
    .withColumn(
        "city",
        F.when(
            F.col("city").isNull(), None)
            .when(
                F.col("city").isin(allowed), F.col("city")
            )
            .otherwise(None)
        )
    )

#Sanity check
display( df_silver.select("city").distinct().show())

In [0]:
# fix case in customer name
df_silver.select('customer_name').distinct().show()

In [0]:
# Title case fix
df_silver = df_silver.withColumn(
    "customer_name",
    F.when(
        F.col("customer_name").isNull(), None
    ).otherwise(
        F.initcap("customer_name")
    )
)

In [0]:
# check customer with null city
df_silver.filter( F.col("city").isNull()).show( truncate=False)


In [0]:
# check customer with null city have any city
null_customer_names = (
    df_silver.filter(F.col("city").isNull())
    .select("customer_name")
    .distinct()
    .collect()
)
null_customer_names = [row.customer_name for row in null_customer_names]

df_silver.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)

In [0]:
# bussiness confirmation note:  city corrections
customer_city_fix = {
    # Sprintx Nutrition
    789403: "New Delhi",

    # Zenathlete Foods
    789420: "Bengaluru",

    # Primefuel Nutrition
    789521: "Hyderabad",

    # Recovery Lane
    789603: "Hyderabad"
}
df_fix = spark.createDataFrame(
    [ (k,v) for k, v in customer_city_fix.items()],
    [ "customer_id", "fixed_city"]
)
display( df_fix)

In [0]:
df_silver = (
    df_silver
    .join( df_fix, "customer_id", "left")
    .withColumn(
        "city",
        F.coalesce( "city", "fixed_city")
    )
    .drop("fixed_city")
)

In [0]:
# schema apply for silver
df_silver = df_silver.withColumn(
    "customer_id",
    F.col("customer_id").cast("string")
)
print( df_silver.printSchema())

### Standardizing Customer Attributes to Match Parent Company Data Model

In [0]:
df_silver = (
    df_silver
    # build final customer column: "CustomerName-City" or "CustomerName-Unknown"
    .withColumn(
        "customer",
        F.concat_ws("-", "customer_name", 
                    F.coalesce(
                        F.col("city"),
                        F.lit("Unknown")
                    ))
    )
    
    #Static attributes aligned with parent data model
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)

In [0]:
display( df_silver.limit(5))

In [0]:
# write to silver
df_silver.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .option("mergeSchema", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

### Gold Processing

In [0]:
df_silver = spark.sql(f"SELECT *    FROM    {catalog}.{silver_schema}.{data_source}    "
    )
df_gold = df_silver.select(
    "customer_id",
    "customer_name",
    "city",
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
df_gold.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

### Merging Data source with parent

In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")
df_child_customer = spark.table("fmcg.gold.sb_dim_customers").select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
# UPSERT . if match then copy source to target, if not match insert
delta_table.alias("target").merge(
    source = df_child_customer.alias("source"),
    condition = "target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
